In [13]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [14]:
# Data and model paths
ROOT = Path.cwd().parent
DATA_DIR = ROOT / 'ml' / 'data'
MODELS_DIR = ROOT / 'include' / 'models'

In [15]:
# Model parameters
EPOCHS = 500
LEARNING_RATE = 0.01

In [16]:
# Load the dataset
train_data = pd.read_csv(DATA_DIR / 'MNIST' / 'pre_train.csv')
test_data = pd.read_csv(DATA_DIR / 'MNIST' / 'test.csv')

print(f'Train data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

Train data shape: (20000, 197)
Test data shape: (10000, 197)


In [17]:
# Extract features and labels
X_train = train_data.drop('label', axis=1).to_numpy()
X_test = test_data.drop('label', axis=1).to_numpy()

y_train = train_data['label'].to_numpy()
y_test = test_data['label'].to_numpy()

print(f'Train Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}'); print()
print(f'Test Class distribution:\n{pd.Series(y_test).value_counts().sort_index()}')

Train Class distribution:
0    1972
1    2250
2    1997
3    2040
4    1950
5    1804
6    1965
7    2084
8    1950
9    1988
Name: count, dtype: int64

Test Class distribution:
0     986
1    1125
2     999
3    1020
4     975
5     902
6     982
7    1042
8     975
9     994
Name: count, dtype: int64


In [18]:
# Convert data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [28]:
# Define the NN model
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(196, 32), nn.ReLU(),   # Input layer (14 x 14 = 196 pixels) to hidden layer 1 (32 neurons)
            nn.Linear(32, 16), nn.ReLU(),    # Hidden layer 1 (32 neurons) to hidden layer 2 (16 neurons)
            nn.Linear(16, 10)                # Hidden layer 2 (16 neurons) to output layer (10 classes)
        )

    def forward(self, x):
        return self.network(x)


model = MNISTNet()

In [29]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [30]:
# Training loop
print("Training...")
for epoch in range(EPOCHS):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1:3d}/{EPOCHS}] "
            f"Loss: {loss.item():.4f}"
        )

Training...
Epoch [ 10/500] Loss: 2.2954
Epoch [ 20/500] Loss: 2.0640
Epoch [ 30/500] Loss: 1.9402
Epoch [ 40/500] Loss: 1.8950
Epoch [ 50/500] Loss: 1.8739
Epoch [ 60/500] Loss: 1.8572
Epoch [ 70/500] Loss: 1.7918
Epoch [ 80/500] Loss: 1.6625
Epoch [ 90/500] Loss: 1.6186
Epoch [100/500] Loss: 1.5837
Epoch [110/500] Loss: 1.5611
Epoch [120/500] Loss: 1.5444
Epoch [130/500] Loss: 1.5294
Epoch [140/500] Loss: 1.5089
Epoch [150/500] Loss: 1.4906
Epoch [160/500] Loss: 1.4786
Epoch [170/500] Loss: 1.4605
Epoch [180/500] Loss: 1.4323
Epoch [190/500] Loss: 1.3997
Epoch [200/500] Loss: 1.3692
Epoch [210/500] Loss: 1.3434
Epoch [220/500] Loss: 1.3192
Epoch [230/500] Loss: 1.2996
Epoch [240/500] Loss: 1.2773
Epoch [250/500] Loss: 1.2592
Epoch [260/500] Loss: 1.2389
Epoch [270/500] Loss: 1.2119
Epoch [280/500] Loss: 1.1864
Epoch [290/500] Loss: 1.1651
Epoch [300/500] Loss: 1.1361
Epoch [310/500] Loss: 1.1105
Epoch [320/500] Loss: 1.0870
Epoch [330/500] Loss: 1.0583
Epoch [340/500] Loss: 1.0398
Ep

In [31]:
# Evaluate the model
model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    y_pred_classes = torch.argmax(y_pred, dim=1)

    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.7148


In [33]:
class MNISTNetExport(nn.Module):
    def __init__(self):
        super(MNISTNetExport, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(196, 32), nn.ReLU(),   # Input layer (14 x 14 = 196 pixels) to hidden layer 1 (32 neurons)
            nn.Linear(32, 16), nn.ReLU(),    # Hidden layer 1 (32 neurons) to hidden layer 2 (16 neurons)
            nn.Linear(16, 10),                # Hidden layer 2 (16 neurons) to output layer (10 classes)
            nn.Softmax(dim=1)                 # Softmax activation for output probabilities
        )

    def forward(self, x):
        return self.network(x)

export_model = MNISTNetExport()
export_model.load_state_dict(model.state_dict())

export_model.eval()
with torch.no_grad():
    y_pred_export = export_model(X_test)
    y_pred_classes_export = torch.argmax(y_pred_export, dim=1)

    accuracy_export = accuracy_score(y_test, y_pred_classes_export)
    print(f"\nTest Accuracy (Export Model): {accuracy_export:.4f}")


Test Accuracy (Export Model): 0.7148


In [34]:
from aifes import pytorch2aifes

pytorch2aifes.convert_to_fnn_f32_express(export_model, str(MODELS_DIR / 'MNISTNet'))